In [1]:
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "mrpc"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3668
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence1'], raw_train_dataset[5]['sentence2']

('Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .',
 "With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .")

In [11]:
raw_train_dataset[5]['label']

1

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 100]),
 'token_type_ids': torch.Size([32, 100]),
 'attention_mask': torch.Size([32, 100])}

# 4. Loading model

In [19]:
import sys
sys.path.insert(0, '../../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(1.2996, grad_fn=<NllLossBackward0>) torch.Size([32, 2])


In [25]:
device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

device

device(type='cuda', index=1)

### Load Trained Weights

In [26]:
load_path = '../../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt', map_location=device)
model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [27]:
from train_eval_func import eval_loop

In [28]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

In [31]:
import torch
from tqdm import tqdm
import pandas as pd
import os 

def generate_glue_submission(
    model,
    test_dataloader,
    task_name,
    output_dir='.',
    output_filename=None,
    device='cuda'
):
    """
    Generate GLUE submission file in TSV format for test set.
    
    This function creates the exact TSV format required for GLUE benchmark submission.
    Each task has specific requirements for column names and prediction format.
    
    Args:
        model: Trained BERT model
        test_dataloader: DataLoader for test set (NO LABELS)
        task_name: GLUE task name (case-insensitive)
            - 'cola', 'sst2', 'mrpc', 'stsb', 'qqp', 'mnli', 'mnli-mm', 
              'qnli', 'rte', 'wnli', 'ax'
        output_dir: Directory to save TSV file
        output_filename: Custom filename (None = auto-generate as {TASK}.tsv)
        device: Device to run inference on
    
    Returns:
        str: Path to generated TSV file
    
    GLUE Submission Format Requirements:
        CoLA:  index, prediction (0/1)
        SST-2: index, prediction (0/1)
        MRPC:  index, prediction (0/1)
        STS-B: index, prediction (0.0-5.0, regression)
        QQP:   index, prediction (0/1)
        MNLI:  index, prediction (entailment/neutral/contradiction)
        QNLI:  index, prediction (entailment/not_entailment)
        RTE:   index, prediction (entailment/not_entailment)
        WNLI:  index, prediction (0/1)
        AX:    index, prediction (entailment/neutral/contradiction)
    
    Examples:
        # Generate MRPC submission
        generate_glue_submission(
            model, test_dataloader, 'mrpc',
            output_dir='./submissions'
        )
        # Creates: ./submissions/MRPC.tsv
        
        # Generate QNLI with custom filename
        generate_glue_submission(
            model, test_dataloader, 'qnli',
            output_dir='./submissions',
            output_filename='QNLI_8layer_merged.tsv'
        )
    """
    model.eval()
    model.to(device)
    
    task_name = task_name.lower()
    
    # Task-specific configurations
    task_configs = {
        'cola': {'num_labels': 2, 'label_type': 'binary', 'label_list': None},
        'sst2': {'num_labels': 2, 'label_type': 'binary', 'label_list': None},
        'mrpc': {'num_labels': 2, 'label_type': 'binary', 'label_list': None},
        'stsb': {'num_labels': 1, 'label_type': 'regression', 'label_list': None},
        'qqp': {'num_labels': 2, 'label_type': 'binary', 'label_list': None},
        'mnli': {'num_labels': 3, 'label_type': 'multiclass', 
                 'label_list': ['entailment', 'neutral', 'contradiction']},
        'mnli-mm': {'num_labels': 3, 'label_type': 'multiclass',
                    'label_list': ['entailment', 'neutral', 'contradiction']},
        'qnli': {'num_labels': 2, 'label_type': 'multiclass',
                 'label_list': ['entailment', 'not_entailment']},
        'rte': {'num_labels': 2, 'label_type': 'multiclass',
                'label_list': ['entailment', 'not_entailment']},
        'wnli': {'num_labels': 2, 'label_type': 'binary', 'label_list': None},
        'ax': {'num_labels': 3, 'label_type': 'multiclass',
               'label_list': ['entailment', 'neutral', 'contradiction']},
    }
    
    if task_name not in task_configs:
        raise ValueError(
            f"Unknown task: {task_name}. "
            f"Supported tasks: {', '.join(task_configs.keys())}"
        )
    
    config = task_configs[task_name]
    
    print(f"Generating GLUE submission for {task_name.upper()}")
    print(f"Task type: {config['label_type']}")
    
    # Generate predictions
    predictions = []
    indices = []
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(test_dataloader, desc="Predicting")):
            # Move batch to device (excluding labels if present)
            batch_inputs = {
                k: v.to(device) for k, v in batch.items() 
                if k not in ['labels', 'label', 'idx', 'index']
            }
            
            # Forward pass
            outputs = model(**batch_inputs)
            logits = outputs.logits
            
            # Get predictions based on task type
            if config['label_type'] == 'regression':
                # STS-B: regression task (0.0 to 5.0)
                preds = logits.squeeze(-1).cpu().numpy()
                # Clip to valid range
                preds = preds.clip(0.0, 5.0)
            else:
                # Classification: get argmax
                preds = torch.argmax(logits, dim=-1).cpu().numpy()
            
            predictions.extend(preds.tolist())
            
            # # Track indices (batch index * batch_size + position in batch)
            # batch_size = len(preds)
            # batch_indices = list(range(batch_idx * batch_size, 
            #                           batch_idx * batch_size + batch_size))
            # indices.extend(batch_indices)

    for i in range(len(predictions)):
        indices.append(i)
    
    # Convert predictions to required format
    if config['label_type'] == 'regression':
        # STS-B: keep as float
        formatted_predictions = predictions
    elif config['label_type'] == 'multiclass' and config['label_list']:
        # MNLI, QNLI, RTE, AX: map to label strings
        formatted_predictions = [config['label_list'][int(p)] for p in predictions]
    else:
        # Binary tasks (CoLA, SST-2, MRPC, QQP, WNLI): keep as 0/1
        formatted_predictions = [int(p) for p in predictions]
    
    # Create DataFrame
    df = pd.DataFrame({
        'index': indices,
        'prediction': formatted_predictions
    })
    
    # Determine output filename
    if output_filename is None:
        output_filename = f"{task_name.upper()}.tsv"
    
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    # Full output path
    output_path = os.path.join(output_dir, output_filename)
    
    # Save to TSV
    df.to_csv(output_path, sep='\t', index=False)
    
    print(f"\n✓ Submission file created: {output_path}")
    print(f"  Total predictions: {len(predictions)}")
    print(f"  Format: index, prediction")
    
    # Show sample
    print(f"\nFirst 5 rows:")
    print(df.head())
    
    return output_path

In [32]:
generate_glue_submission(model, eval_dataloader, task_name, output_dir='.', device=device)

Generating GLUE submission for MRPC
Task type: binary


Predicting: 100%|██████████| 54/54 [00:06<00:00,  7.78it/s]


✓ Submission file created: ./MRPC.tsv
  Total predictions: 1725
  Format: index, prediction

First 5 rows:
   index  prediction
0      0           1
1      1           1
2      2           1
3      3           1
4      4           0


'./MRPC.tsv'

# 5. Submission Preparation

In [29]:
from train_eval_func import generate_glue_submission

In [30]:
generate_glue_submission(model, eval_dataloader, task_name, output_dir='.', device=device)

Generating GLUE submission for MRPC
Task type: binary


Predicting: 100%|██████████| 54/54 [00:02<00:00, 22.36it/s]


✓ Submission file created: ./MRPC.tsv
  Total predictions: 1725
  Format: index, prediction

First 5 rows:
   index  prediction
0      0           1
1      1           1
2      2           1
3      3           1
4      4           0


'./MRPC.tsv'